# Music Recommender System: Math-Based Scoring Rules

This notebook demonstrates a complete implementation of a music recommendation scoring system that combines energy and mood preferences using mathematical formulas.

**Core Components:**
- Energy score: Measures closeness to target energy (0-1 scale)
- Mood score: Categorical matching with partial credit for related moods
- Combined score: Weighted addition (60% energy, 40% mood)

## 1. Data Models: Song and UserProfile

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple
import math

@dataclass
class Song:
    """Represents a song with energy and mood attributes."""
    id: int
    title: str
    artist: str
    energy: float  # [0.0, 1.0]
    mood: str      # categorical: "happy", "chill", "intense", "relaxed", "moody", "focused"

@dataclass
class UserProfile:
    """Represents a user's music preferences."""
    user_id: int
    favorite_mood: str     # categorical
    target_energy: float   # [0.0, 1.0]

# Create example instances
song1 = Song(id=1, title="Upbeat Morning", artist="Joy Makers", energy=0.8, mood="happy")
song2 = Song(id=2, title="Study Focus", artist="Lofi Beats", energy=0.4, mood="chill")
song3 = Song(id=3, title="Rock Anthem", artist="Loud Band", energy=0.95, mood="intense")

user = UserProfile(user_id=101, favorite_mood="happy", target_energy=0.8)

print("Example Songs:")
print(f"  {song1}")
print(f"  {song2}")
print(f"  {song3}")
print(f"\nExample User: {user}")

## 2. Energy Scoring: Mathematical Formulation

### Option A: Absolute Difference Penalty (Linear)

$$\text{energy\_score} = 1 - |\text{song.energy} - \text{user.target\_energy}|$$

- **Range:** [0, 1]
- **Characteristics:** Simple, intuitive, linear penalty
- **When to use:** General recommendations where all distances are equally important

### Option B: Squared Difference Penalty (Quadratic)

$$\text{energy\_score} = 1 - (\text{song.energy} - \text{user.target\_energy})^2$$

- **Range:** [0, 1]
- **Characteristics:** Creates a "sweet spot," more forgiving of small differences, harsher on large ones
- **When to use:** When you want stricter matching around the target energy

### Normalization

Both formulas are self-normalizing:
- When $|\text{difference}| = 0$ (perfect match): score = 1.0
- When $|\text{difference}| = 1$ (maximum distance): score = 0.0
- Both are always in [0, 1] by construction

## 3. Energy Scoring: Python Implementation

In [ ]:
def score_energy_linear(song_energy: float, target_energy: float) -> float:
    """
    Score energy using absolute difference (linear penalty).
    
    Formula: energy_score = 1 - |song_energy - target_energy|
    
    Args:
        song_energy: Song's energy level [0.0, 1.0]
        target_energy: User's target energy level [0.0, 1.0]
        
    Returns:
        Score in [0.0, 1.0] where 1.0 is perfect match
    """
    return 1.0 - abs(song_energy - target_energy)


def score_energy_quadratic(song_energy: float, target_energy: float) -> float:
    """
    Score energy using squared difference (quadratic penalty).
    
    Formula: energy_score = 1 - (song_energy - target_energy)^2
    
    More forgiving for small differences, harsher for large ones.
    
    Args:
        song_energy: Song's energy level [0.0, 1.0]
        target_energy: User's target energy level [0.0, 1.0]
        
    Returns:
        Score in [0.0, 1.0] where 1.0 is perfect match
    """
    diff = song_energy - target_energy
    return 1.0 - (diff * diff)


# Test with various cases
test_cases = [
    (0.8, 0.8, "Perfect match"),
    (0.75, 0.8, "Close (0.05 difference)"),
    (0.6, 0.8, "Moderate (0.2 difference)"),
    (0.2, 0.8, "Far (0.6 difference)"),
    (0.0, 1.0, "Maximum distance"),
]

print("Energy Scoring Comparison (Linear vs Quadratic)\n")
print(f"{'Song Energy':<12} {'Target':<8} {'Difference':<12} {'Linear':<10} {'Quadratic':<10} {'Case':<25}")
print("-" * 85)

for song_e, target_e, label in test_cases:
    linear = score_energy_linear(song_e, target_e)
    quad = score_energy_quadratic(song_e, target_e)
    diff = abs(song_e - target_e)
    print(f"{song_e:<12.2f} {target_e:<8.2f} {diff:<12.2f} {linear:<10.4f} {quad:<10.4f} {label:<25}")

## 4. Mood Scoring: Similarity Matrix and Formulation

### Mood Similarity Matrix

A **6×6 similarity matrix** for the six mood categories:

|         | happy | chill | intense | relaxed | moody | focused |
|---------|-------|-------|---------|---------|-------|----------|
| **happy**   | **1.0** | 0.0   | 0.0     | 0.0     | 0.0   | 0.6     |
| **chill**   | 0.0   | **1.0** | 0.0     | 0.8     | 0.0   | 0.0     |
| **intense** | 0.0   | 0.0   | **1.0**     | 0.0     | 0.5   | 0.0     |
| **relaxed** | 0.0   | 0.8   | 0.0     | **1.0**     | 0.0   | 0.0     |
| **moody**   | 0.0   | 0.0   | 0.5     | 0.0     | **1.0** | 0.0     |
| **focused** | 0.6   | 0.0   | 0.0     | 0.0     | 0.0   | **1.0**     |

### Logic

**Exact matches (diagonal = 1.0):**
- Same mood gets full credit

**Related moods (0.8 similarity):**
- "chill" ↔ "relaxed" (both calming, introspective)

**Somewhat related moods (0.6 similarity):**
- "happy" ↔ "focused" (both positive, goal-oriented)

**Loosely related moods (0.5 similarity):**
- "intense" ↔ "moody" (both high emotion/energy)

**Unrelated moods (0.0):**
- All other pairs get no credit (e.g., "happy" ↔ "chill", "intense" ↔ "relaxed")

### Formula

$$\text{mood\_score} = \text{similarity\_matrix}[\text{song.mood}][\text{user.favorite\_mood}]$$

## 5. Mood Scoring: Python Implementation

In [ ]:
# Define the mood similarity matrix
MOOD_SIMILARITY = {
    # Exact matches (diagonal)
    ("happy", "happy"): 1.0,
    ("chill", "chill"): 1.0,
    ("intense", "intense"): 1.0,
    ("relaxed", "relaxed"): 1.0,
    ("moody", "moody"): 1.0,
    ("focused", "focused"): 1.0,
    
    # Related calm moods: chill ↔ relaxed (0.8 similarity)
    ("chill", "relaxed"): 0.8,
    ("relaxed", "chill"): 0.8,
    
    # Related energetic moods: happy ↔ focused (0.6 similarity)
    ("happy", "focused"): 0.6,
    ("focused", "happy"): 0.6,
    
    # Intense moods somewhat related: intense ↔ moody (0.5 similarity)
    ("intense", "moody"): 0.5,
    ("moody", "intense"): 0.5,
    
    # All unspecified pairs get 0.0 (no credit for unrelated moods)
}


def get_mood_similarity(song_mood: str, favorite_mood: str) -> float:
    """
    Look up mood similarity from the similarity matrix.
    
    Args:
        song_mood: The song's mood
        favorite_mood: The user's favorite mood
        
    Returns:
        Similarity score in [0.0, 1.0]
    """
    return MOOD_SIMILARITY.get((song_mood, favorite_mood), 0.0)


def score_mood(song_mood: str, favorite_mood: str) -> float:
    """
    Score mood based on similarity to user's favorite mood.
    
    Args:
        song_mood: The song's mood (string)
        favorite_mood: The user's favorite mood (string)
        
    Returns:
        Score in [0.0, 1.0]
    """
    return get_mood_similarity(song_mood, favorite_mood)


# Test mood scoring
all_moods = ["happy", "chill", "intense", "relaxed", "moody", "focused"]
user_mood = "happy"

print(f"Mood Scores for User Preference: '{user_mood}'\n")
print(f"{'Song Mood':<12} {'Score':<8} {'Description':<40}")
print("-" * 60)

for mood in all_moods:
    score = score_mood(mood, user_mood)
    if score == 1.0:
        desc = "Exact match"
    elif score > 0.5:
        desc = "Related mood (partial credit)"
    elif score > 0.0:
        desc = "Loosely related mood"
    else:
        desc = "Unrelated mood (no credit)"
    print(f"{mood:<12} {score:<8.1f} {desc:<40}")

## 6. Combined Scoring: Weighted Addition

### Formula

$$\text{total\_score} = (\text{energy\_weight} \times \text{energy\_score}) + (\text{mood\_weight} \times \text{mood\_score})$$

### Default Weights (60/40 split)

$$\text{total\_score} = (0.6 \times \text{energy\_score}) + (0.4 \times \text{mood\_score})$$

### Normalization

Since:
- $0 \leq \text{energy\_score} \leq 1$
- $0 \leq \text{mood\_score} \leq 1$
- Weights sum to 1.0: $0.6 + 0.4 = 1.0$

Therefore:
$$0 \leq \text{total\_score} \leq 1$$

**Result:** Final score is always in [0, 1] ✓

## 7. Complete Scoring Function with Examples

In [ ]:
def score_song(
    song: Song,
    user: UserProfile,
    energy_weight: float = 0.6,
    mood_weight: float = 0.4,
    energy_method: str = "quadratic"
) -> Tuple[float, float, float, float]:
    """
    Calculate a comprehensive recommendation score for a song given a user profile.
    
    Formula (default weights):
        total_score = (0.6 × energy_score) + (0.4 × mood_score)
    
    Args:
        song: A Song object with 'energy' (float) and 'mood' (str) fields
        user: A UserProfile object with 'target_energy' (float) and 'favorite_mood' (str) fields
        energy_weight: Weight for energy component (default 0.6)
        mood_weight: Weight for mood component (default 0.4)
        energy_method: "linear" or "quadratic" scoring for energy (default "quadratic")
        
    Returns:
        Tuple of (total_score, energy_score, mood_score, combined_normalized_score)
        Each value is in [0.0, 1.0]
        
    Raises:
        ValueError: If energy_method is not "linear" or "quadratic"
        AssertionError: If weights don't sum to 1.0
    """
    # Validate weights
    assert abs(energy_weight + mood_weight - 1.0) < 1e-6, \
        f"Weights must sum to 1.0, got {energy_weight + mood_weight}"
    
    if energy_method not in ("linear", "quadratic"):
        raise ValueError(f"energy_method must be 'linear' or 'quadratic', got '{energy_method}'")
    
    # Calculate component scores
    if energy_method == "linear":
        energy_score = score_energy_linear(song.energy, user.target_energy)
    else:  # quadratic
        energy_score = score_energy_quadratic(song.energy, user.target_energy)
    
    mood_score = score_mood(song.mood, user.favorite_mood)
    
    # Weighted combination
    total_score = (energy_weight * energy_score) + (mood_weight * mood_score)
    
    return total_score, energy_score, mood_score, total_score


# Test scenarios
print("\n" + "="*90)
print("SCENARIO 1: Perfect Match")
print("="*90)

perfect_song = Song(id=1, title="Perfect Energy & Mood", artist="Ideal Artist", 
                     energy=0.8, mood="happy")
user = UserProfile(user_id=1, favorite_mood="happy", target_energy=0.8)

total, energy, mood, _ = score_song(perfect_song, user)
print(f"Song: {perfect_song.title}")
print(f"User: target_energy={user.target_energy}, favorite_mood='{user.favorite_mood}'")
print(f"  Energy Score:     {energy:.4f}")
print(f"  Mood Score:       {mood:.4f}")
print(f"  Combined Score:   {total:.4f}")
print(f"  → Result: {total*100:.1f}% match")

print("\n" + "="*90)
print("SCENARIO 2: Energy Mismatch, Mood Match")
print("="*90)

energy_mismatch = Song(id=2, title="Wrong Energy", artist="Artist 2", 
                        energy=0.2, mood="happy")

total, energy, mood, _ = score_song(energy_mismatch, user)
print(f"Song: {energy_mismatch.title}")
print(f"User: target_energy={user.target_energy}, favorite_mood='{user.favorite_mood}'")
print(f"  Energy Score:     {energy:.4f}  (0.2 vs 0.8 target)")
print(f"  Mood Score:       {mood:.4f}  (happy matches happy)")
print(f"  Combined Score:   {total:.4f}")
print(f"  → Result: {total*100:.1f}% match")

print("\n" + "="*90)
print("SCENARIO 3: Energy Match, Related Mood")
print("="*90)

mood_related = Song(id=3, title="Related Mood", artist="Artist 3", 
                     energy=0.8, mood="focused")

total, energy, mood, _ = score_song(mood_related, user)
print(f"Song: {mood_related.title}")
print(f"User: target_energy={user.target_energy}, favorite_mood='{user.favorite_mood}'")
print(f"  Energy Score:     {energy:.4f}  (perfect 0.8)")
print(f"  Mood Score:       {mood:.4f}  (focused ~ happy, partial credit)")
print(f"  Combined Score:   {total:.4f}")
print(f"  → Result: {total*100:.1f}% match")

print("\n" + "="*90)
print("SCENARIO 4: Worst Case (Maximum Mismatch)")
print("="*90)

worst_song = Song(id=4, title="Opposite Everything", artist="Artist 4", 
                   energy=0.1, mood="intense")

total, energy, mood, _ = score_song(worst_song, user)
print(f"Song: {worst_song.title}")
print(f"User: target_energy={user.target_energy}, favorite_mood='{user.favorite_mood}'")
print(f"  Energy Score:     {energy:.4f}  (0.1 vs 0.8, far apart)")
print(f"  Mood Score:       {mood:.4f}  (intense vs happy, unrelated)")
print(f"  Combined Score:   {total:.4f}")
print(f"  → Result: {total*100:.1f}% match")

## 8. Visualize Score Distribution Across a Song Library

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a diverse song library
song_library = [
    Song(id=1, title="Sunrise City", artist="Neon Echo", energy=0.82, mood="happy"),
    Song(id=2, title="Midnight Coding", artist="LoRoom", energy=0.42, mood="chill"),
    Song(id=3, title="Storm Runner", artist="Voltline", energy=0.91, mood="intense"),
    Song(id=4, title="Library Rain", artist="Paper Lanterns", energy=0.35, mood="chill"),
    Song(id=5, title="Gym Hero", artist="Max Pulse", energy=0.93, mood="intense"),
    Song(id=6, title="Spacewalk Thoughts", artist="Orbit Bloom", energy=0.28, mood="chill"),
    Song(id=7, title="Coffee Shop Stories", artist="Slow Stereo", energy=0.37, mood="relaxed"),
    Song(id=8, title="Night Drive Loop", artist="Neon Echo", energy=0.75, mood="moody"),
    Song(id=9, title="Focus Flow", artist="LoRoom", energy=0.40, mood="focused"),
    Song(id=10, title="Rooftop Lights", artist="Indigo Parade", energy=0.76, mood="happy"),
]

# Test user
user = UserProfile(user_id=101, favorite_mood="happy", target_energy=0.8)

# Score all songs
scores = []
for song in song_library:
    score, _, _, _ = score_song(song, user, energy_method="quadratic")
    scores.append(score)

# Display ranked results
print("\nSong Library Ranked by Recommendation Score")
print(f"(User: target_energy={user.target_energy}, favorite_mood='{user.favorite_mood}')\n")
print(f"{'Rank':<5} {'Score':<8} {'Title':<25} {'Artist':<20} {'Energy':<8} {'Mood':<10}")
print("-" * 80)

ranked = sorted(zip(scores, song_library), key=lambda x: x[0], reverse=True)
for rank, (score, song) in enumerate(ranked, 1):
    print(f"{rank:<5} {score:<8.4f} {song.title:<25} {song.artist:<20} {song.energy:<8.2f} {song.mood:<10}")

### Score Distribution Visualization

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Music Recommender Analysis\n(User: {user.favorite_mood} mood, {user.target_energy} energy target)', 
             fontsize=14, fontweight='bold')

# Plot 1: Score Distribution (Histogram)
ax1 = axes[0, 0]
ax1.hist(scores, bins=8, color='steelblue', edgecolor='black', alpha=0.7)
ax1.set_xlabel('Recommendation Score')
ax1.set_ylabel('Number of Songs')
ax1.set_title('Score Distribution')
ax1.set_xlim(0, 1)
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Energy vs Score
ax2 = axes[0, 1]
energies = [song.energy for song in song_library]
colors = ['green' if song.mood == user.favorite_mood else 'orange' for song in song_library]
ax2.scatter(energies, scores, s=100, c=colors, alpha=0.6, edgecolors='black')
ax2.axvline(user.target_energy, color='red', linestyle='--', linewidth=2, label='Target Energy')
ax2.set_xlabel('Song Energy')
ax2.set_ylabel('Recommendation Score')
ax2.set_title('Energy vs Score')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(alpha=0.3)

# Plot 3: Mood Category Distribution
ax3 = axes[1, 0]
moods = [song.mood for song in song_library]
mood_counts = {}
for mood in moods:
    mood_counts[mood] = mood_counts.get(mood, 0) + 1
ax3.bar(mood_counts.keys(), mood_counts.values(), color='coral', edgecolor='black', alpha=0.7)
ax3.set_xlabel('Mood Category')
ax3.set_ylabel('Number of Songs')
ax3.set_title('Song Distribution by Mood')
ax3.grid(axis='y', alpha=0.3)
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Plot 4: Top 5 Recommendations
ax4 = axes[1, 1]
top_5 = ranked[:5]
titles = [song.title[:15] for _, song in top_5]
top_scores = [score for score, _ in top_5]
colors_top = ['#2ecc71' if score > 0.7 else '#3498db' if score > 0.5 else '#e74c3c' for score in top_scores]
bars = ax4.barh(titles, top_scores, color=colors_top, edgecolor='black', alpha=0.7)
ax4.set_xlabel('Recommendation Score')
ax4.set_title('Top 5 Recommendations')
ax4.set_xlim(0, 1)
for i, (bar, score) in enumerate(zip(bars, top_scores)):
    ax4.text(score + 0.02, i, f'{score:.3f}', va='center', fontsize=9)
ax4.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nVisualization complete!")

## Summary: Key Takeaways

### Energy Scoring
- **Linear:** Simple distance penalty, good for general use
- **Quadratic:** Creates "sweet spot," more selective
- Both normalized to [0, 1]

### Mood Scoring
- Exact match: 1.0
- Related moods: 0.5-0.8 (partial credit)
- Unrelated moods: 0.0 (no credit)

### Combined Score
- Weighted addition: 60% energy + 40% mood
- Ensures final score is in [0, 1]
- Energy dominates, but mood still significant

### When to Use
- **Music streaming:** Personalized playlists
- **Fitness apps:** Match workout intensity to song energy
- **Mood-based apps:** Help users find music matching their emotional state